# Campaign Efficiency Analysis - Dorje Teas

[SYNTHETIC DATA - Illustrative only. No internal Dorje data used.]

Purpose: evaluate paid campaign efficiency using ROAS, contribution ROAS, CAC, CTR, CVR, and decision rules that avoid scaling unprofitable activity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

DATA_DIR = Path('../04_data_model')
if not DATA_DIR.exists():
    DATA_DIR = Path('04_data_model')
assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR.resolve()}'

def rupee(value):
    return f'Rs. {value:,.0f}'

ad = pd.read_csv(DATA_DIR / 'sample_ad_spend.csv')
ad.head()

## Load and Validate Campaign Data

In [ ]:
required = {'week','month','platform','campaign_id','campaign_name','spend','impressions','clicks','ctr_pct','sessions','orders','cvr_pct','revenue','new_customers','cac','roas','contribution_roas'}
missing = required - set(ad.columns)
assert not missing, missing
assert (ad['spend'] > 0).all()
assert ((ad['roas'] - ad['revenue'] / ad['spend']).abs() <= 0.1).all()
assert ad['contribution_roas'].between(0.3, 5.0).all()
ad[['spend','revenue','roas','contribution_roas','cac','cvr_pct']].describe()

## Campaign-Level Aggregation

In [ ]:
ad['contribution_value'] = ad['contribution_roas'] * ad['spend']
campaign = ad.groupby(['campaign_id','campaign_name'], as_index=False).agg(
    total_spend=('spend','sum'),
    total_revenue=('revenue','sum'),
    contribution_value=('contribution_value','sum'),
    new_customers=('new_customers','sum'),
    orders=('orders','sum'),
    sessions=('sessions','sum'),
    avg_ctr_pct=('ctr_pct','mean')
)
campaign['blended_roas'] = campaign['total_revenue'] / campaign['total_spend']
campaign['contribution_roas'] = campaign['contribution_value'] / campaign['total_spend']
campaign['avg_cac'] = campaign['total_spend'] / campaign['new_customers'].replace(0, np.nan)
campaign['avg_cvr_pct'] = campaign['orders'] / campaign['sessions'] * 100
campaign.sort_values('contribution_roas', ascending=False)

## ROAS vs Contribution ROAS Comparison

In [ ]:
plot_df = campaign.melt(id_vars='campaign_id', value_vars=['blended_roas','contribution_roas'], var_name='metric', value_name='value')
ax = sns.barplot(data=plot_df, x='campaign_id', y='value', hue='metric')
ax.axhline(1.0, color='black', linestyle='--', label='Contribution ROAS = 1.0')
ax.set_title('Dorje Teas ROAS vs Contribution ROAS [SYNTHETIC DATA]')
ax.set_xlabel('Campaign ID')
ax.set_ylabel('Efficiency multiple')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## CAC Trend Over Time by Campaign

In [ ]:
trend = ad.pivot_table(index='week', columns='campaign_id', values='cac', aggfunc='mean')
ax = trend.plot(figsize=(12,6))
ax.set_title('Dorje Teas CAC Trend by Campaign [SYNTHETIC DATA]')
ax.set_xlabel('Week')
ax.set_ylabel('CAC (Rs.)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## CTR vs CVR Diagnostic Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
sns.scatterplot(data=campaign, x='avg_ctr_pct', y='avg_cvr_pct', size='total_spend', hue='contribution_roas', palette='viridis', sizes=(80,500), ax=ax)
for _, row in campaign.iterrows():
    ax.text(row['avg_ctr_pct'], row['avg_cvr_pct'], row['campaign_id'], fontsize=8, ha='left', va='bottom')
ax.set_title('Dorje Teas CTR vs CVR Diagnostic [SYNTHETIC DATA]')
ax.set_xlabel('Average CTR (%)')
ax.set_ylabel('Average CVR (%)')
plt.tight_layout()
plt.show()

## Weekly Spend Efficiency Trend

In [ ]:
weekly = ad.groupby('week', as_index=False).agg(spend=('spend','sum'), revenue=('revenue','sum'), contribution_value=('contribution_value','sum'))
weekly['roas'] = weekly['revenue'] / weekly['spend']
weekly['contribution_roas'] = weekly['contribution_value'] / weekly['spend']
ax = sns.lineplot(data=weekly.melt(id_vars='week', value_vars=['roas','contribution_roas'], var_name='metric', value_name='value'), x='week', y='value', hue='metric')
ax.axhline(1.0, color='black', linestyle='--')
ax.set_title('Dorje Teas Weekly Spend Efficiency Trend [SYNTHETIC DATA]')
ax.set_xlabel('Week')
ax.set_ylabel('Efficiency multiple')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## Campaign Decision Table

In [ ]:
def decide(row):
    if row['contribution_roas'] >= 1.5 and row['avg_cvr_pct'] >= 2.0:
        return 'SCALE'
    if row['contribution_roas'] >= 1.0 and row['avg_cvr_pct'] >= 1.5:
        return 'HOLD'
    if row['contribution_roas'] >= 0.8 and row['avg_cvr_pct'] >= 1.0:
        return 'FIX'
    return 'PAUSE'

def reasoning(row):
    if row['decision'] == 'SCALE':
        return 'Contribution ROAS and CVR both clear scale thresholds; expand carefully while watching CM.'
    if row['decision'] == 'HOLD':
        return 'Economics are acceptable but not strong enough for aggressive budget expansion.'
    if row['decision'] == 'FIX':
        return 'There is some signal, but landing page, creative, or offer must improve before more spend.'
    return 'Economics are too weak; stop spend until contribution margin or conversion is repaired.'

campaign_decisions = campaign.copy()
campaign_decisions['decision'] = campaign_decisions.apply(decide, axis=1)
campaign_decisions['reasoning'] = campaign_decisions.apply(reasoning, axis=1)
decision_table = campaign_decisions[['campaign_id','campaign_name','total_spend','total_revenue','blended_roas','contribution_roas','avg_cac','avg_cvr_pct','decision','reasoning']].sort_values(['decision','contribution_roas'], ascending=[True, False])
decision_table

## Key Findings

- ROAS and contribution ROAS diverge because ROAS does not account for product cost, packaging, shipping, payment fees, or margin pressure.
- Campaigns with strong CTR but weak CVR should be fixed at the landing page or offer level before budget increases.
- Contribution ROAS below 1.0 is not a scale signal even when reported ROAS looks acceptable.
- First Flush Darjeeling and gifting campaigns should be read through seasonality, not blended with always-on daily tea demand.

## Operator Decisions This Analysis Supports

- SCALE means expand budget or landing page traffic carefully while monitoring contribution margin.
- HOLD means keep running while monitoring CAC, CVR, and contribution ROAS.
- FIX means keep the signal but improve landing page, creative, or offer.
- PAUSE means stop spend until economics are repaired.

## What Real Dorje Data Would Improve

- Actual platform attribution would reduce uncertainty around campaign revenue.
- Actual Shopify conversion path data would identify whether weak CVR is page, cart, or checkout driven.
- Actual product margin by campaign would improve contribution ROAS precision.